## PREPROCESSING- TRAIN -A SET

In [6]:
import os
import xml.etree.ElementTree as ET
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

# ============================================================
# PATH CONFIGURATION
# ============================================================
ICDAR_ROOT = "/home/mca/Shweta/handwritten text detection/dataset/icdar2017"
TRAIN_A_DIR = os.path.join(ICDAR_ROOT, "Train-A")
OUTPUT_BASE = os.path.join(ICDAR_ROOT, "Train-A_Split_Data")

# Namespace used in the PAGE XML format
namespaces = {'ns': 'http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15'}

# ============================================================
# 1. IDENTIFY ALL VALID PAGES
# ============================================================
page_basenames = []
for file in os.listdir(TRAIN_A_DIR):
    if file.endswith(".xml"):
        base = os.path.splitext(file)[0]
        img_path = os.path.join(TRAIN_A_DIR, f"{base}.jpg")
        if os.path.exists(img_path):
            page_basenames.append(base)

page_basenames.sort()
print(f"Total parent page images found: {len(page_basenames)}")

# ============================================================
# 2. STRATIFIED PAGE-LEVEL SPLIT (PREVENTS DATA LEAKAGE)
# ============================================================
train_pages, temp_pages = train_test_split(
    page_basenames, test_size=0.28, random_state=42
)
val_pages, test_pages = train_test_split(
    temp_pages, test_size=0.50, random_state=42
)

splits = {
    "train": train_pages,
    "val": val_pages,
    "test": test_pages
}

print(f"Page Allocation -> Train: {len(train_pages)} | Val: {len(val_pages)} | Test: {len(test_pages)}")

# ============================================================
# 3. PARSE AND EXTRACT HIERARCHICALLY
# ============================================================
for split_name, pages in splits.items():
    print(f"\nProcessing line extraction for [{split_name.upper()}] split...")
    
    # Setup split-specific output folder
    split_img_dir = os.path.join(OUTPUT_BASE, f"{split_name}_lines")
    os.makedirs(split_img_dir, exist_ok=True)
    
    manifest_entries = []
    
    for base_id in pages:
        img_path = os.path.join(TRAIN_A_DIR, f"{base_id}.jpg")
        xml_path = os.path.join(TRAIN_A_DIR, f"{base_id}.xml")
        
        page_img = cv2.imread(img_path)
        if page_img is None:
            continue
            
        # Robust XML Reading Strategy to overcome ParseErrors
        try:
            # First attempt: Try reading strictly
            tree = ET.parse(xml_path)
            root = tree.getroot()
        except ET.ParseError:
            # Fallback Strategy: Read raw contents and sanitize unescaped ampersands
            with open(xml_path, "r", encoding="utf-8", errors="ignore") as f:
                raw_xml = f.read()
            
            # Simple sanitization fix for unescaped characters causing crashes
            sanitized_xml = raw_xml.replace(" & ", " &amp; ")
            try:
                root = ET.fromstring(sanitized_xml)
            except ET.ParseError as final_err:
                print(f"Skipping page {base_id}.xml entirely due to catastrophic syntax issues: {final_err}")
                continue
        
        # Pull text line markers
        lines = root.findall('.//ns:TextLine', namespaces)
        if not lines:  # Fallback to TextRegion
            lines = root.findall('.//ns:TextRegion', namespaces)
            
        for idx, line in enumerate(lines):
            coords_node = line.find('ns:Coords', namespaces)
            text_node = line.find('.//ns:Unicode', namespaces)
            
            if coords_node is not None and text_node is not None and text_node.text:
                transcription = text_node.text.strip()
                points_str = coords_node.attrib.get('points', '')
                
                # Parse points string down into a coordinate matrix
                pts = []
                for pair in points_str.split():
                    if ',' in pair:
                        try:
                            pts.append(list(map(int, pair.split(','))))
                        except ValueError:
                            continue
                if not pts:
                    continue
                    
                pts = np.array(pts, dtype=np.int32)
                x, y, w, h = cv2.boundingRect(pts)
                
                # Add light padding buffer around lines
                pad_y = int(h * 0.04)
                pad_x = int(w * 0.01)
                
                img_h, img_w, _ = page_img.shape
                y1, y2 = max(0, y - pad_y), min(img_h, y + h + pad_y)
                x1, x2 = max(0, x - pad_x), min(img_w, x + w + pad_x)
                
                line_crop = page_img[y1:y2, x1:x2]
                if line_crop.size == 0:
                    continue
                
                # Write individual segmented line image directly to split directory
                crop_filename = f"{base_id}_l{idx}.jpg"
                cv2.imwrite(os.path.join(split_img_dir, crop_filename), line_crop)
                
                # Save the relative path mapping entry for the DataLoader manifest
                manifest_entries.append(f"{split_name}_lines/{crop_filename}\t{transcription}")
                
    # Save the manifest text file for this specific split step
    manifest_path = os.path.join(OUTPUT_BASE, f"{split_name}.txt")
    with open(manifest_path, "w", encoding="utf-8") as f:
        for entry in manifest_entries:
            f.write(f"{entry}\n")
            
    print(f"Saved {len(manifest_entries)} lines into {manifest_path}")

print("\nProcessing Complete! Your data is isolated, leaked-proofed, and ready.")

Total parent page images found: 50
Page Allocation -> Train: 35 | Val: 7 | Test: 8

Processing line extraction for [TRAIN] split...
Skipping page 000028.xml entirely due to catastrophic syntax issues: not well-formed (invalid token): line 20, column 65
Skipping page 000006.xml entirely due to catastrophic syntax issues: not well-formed (invalid token): line 202, column 15
Skipping page 000022.xml entirely due to catastrophic syntax issues: not well-formed (invalid token): line 90, column 125
Skipping page 000041.xml entirely due to catastrophic syntax issues: not well-formed (invalid token): line 165, column 30
Skipping page 000011.xml entirely due to catastrophic syntax issues: not well-formed (invalid token): line 108, column 30
Skipping page 000039.xml entirely due to catastrophic syntax issues: not well-formed (invalid token): line 244, column 30
Saved 793 lines into /home/mca/Shweta/handwritten text detection/dataset/icdar2017/Train-A_Split_Data/train.txt

Processing line extracti

## TRAIN

In [10]:
# ============================================================
# HVLT FOR ICDAR 2017 HISTORICAL LINE RECOGNITION
# PRODUCTION ROBUST TRAINING NOTEBOOK (LOSS-BASED EARLY STOPPING)
# ============================================================

import os
import cv2
import time
import random
import unicodedata
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.amp import autocast, GradScaler

import timm
from transformers import XLMRobertaModel
from jiwer import wer

# ============================================================
# CONFIG
# ============================================================

ROOT_DIR = "/home/mca/Shweta/handwritten text detection/dataset/icdar2017/Train-A_Split_Data"

TRAIN_TXT = os.path.join(ROOT_DIR, "train.txt")
VAL_TXT   = os.path.join(ROOT_DIR, "val.txt")
TEST_TXT  = os.path.join(ROOT_DIR, "test.txt")

IMG_HEIGHT = 32
IMG_WIDTH  = 192  

BATCH_SIZE = 32
LR = 5e-5
MAX_EPOCHS = 100  
MAX_SEQ_LEN = 60  

# Early Stopping Parameter
PATIENCE = 20

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 4
SEED = 42
USE_AMP = True

# ============================================================
# SEED
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ============================================================
# LATIN / GERMAN HISTORICAL VOCABULARY
# ============================================================

HISTORICAL_CHARS = (
    "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "äöüÄÖÜß"  
    "0123456789"
    ".,!?()[]{}:;-_'/\\&%$#@+*=\"„“"
    " "
)

SPECIAL_TOKENS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
ALL_TOKENS = SPECIAL_TOKENS + list(HISTORICAL_CHARS)

char2idx = {c: i for i, c in enumerate(ALL_TOKENS)}
idx2char = {i: c for c, i in char2idx.items()}

VOCAB_SIZE = len(ALL_TOKENS)

PAD_IDX = char2idx["<PAD>"]
SOS_IDX = char2idx["<SOS>"]
EOS_IDX = char2idx["<EOS>"]
UNK_IDX = char2idx["<UNK>"]

print("VOCAB SIZE:", VOCAB_SIZE)

# ============================================================
# DATA PARSING ENGINE
# ============================================================

def load_split_samples(manifest_file, root_base_dir):
    samples = []
    if not os.path.exists(manifest_file):
        print(f"Error: Manifest path missing: {manifest_file}")
        return samples
        
    with open(manifest_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if len(line) == 0:
                continue
            parts = line.split("\t")
            if len(parts) != 2:
                continue
                
            rel_path, text = parts
            text = unicodedata.normalize("NFC", text.strip())
            img_path = os.path.join(root_base_dir, rel_path)
            
            if os.path.exists(img_path):
                samples.append((img_path, text))
    return samples

train_samples = load_split_samples(TRAIN_TXT, ROOT_DIR)
val_samples   = load_split_samples(VAL_TXT, ROOT_DIR)
test_samples  = load_split_samples(TEST_TXT, ROOT_DIR)

print("TRAIN SEGMENTS:", len(train_samples))
print("VAL SEGMENTS:  ", len(val_samples))
print("TEST SEGMENTS: ", len(test_samples))

# ============================================================
# REGULARIZED IMAGE PREPROCESS WITH AUGMENTATIONS
# ============================================================

def preprocess_image(img_path, augment=False):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Apply data augmentation transformations on-the-fly to combat tiny datasets
    if augment:
        # 1. Random Brightness & Contrast
        if random.random() < 0.5:
            alpha = random.uniform(0.8, 1.2)
            beta = random.randint(-15, 15)
            img = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)
        
        # 2. Light Blur/Sharpness Distortions
        if random.random() < 0.3:
            kernel_size = random.choice([3, 5])
            if random.random() < 0.5:
                img = cv2.GaussianBlur(img, (kernel_size, kernel_size), 0)
            else:
                kernel = np.array([[-1,-1,-1], [-1,9,-1], [-1,-1,-1]])
                img = cv2.filter2D(img, -1, kernel)

    h, w = img.shape[:2]
    scale = IMG_HEIGHT / h
    new_w = int(w * scale)
    
    if new_w < 1:
        new_w = 1

    img = cv2.resize(img, (new_w, IMG_HEIGHT))

    if new_w < IMG_WIDTH:
        pad = np.ones((IMG_HEIGHT, IMG_WIDTH - new_w, 3), dtype=np.uint8) * 255
        img = np.concatenate([img, pad], axis=1)
    else:
        img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))

    img = img.astype(np.float32) / 255.0
    img = (img - 0.5) / 0.5
    img = np.transpose(img, (2, 0, 1))

    return torch.tensor(img, dtype=torch.float32)

# ============================================================
# TEXT TRANSFORMATION UTILS
# ============================================================

def encode_text(text):
    tokens = [SOS_IDX]
    for ch in text:
        tokens.append(char2idx.get(ch, UNK_IDX))
    tokens.append(EOS_IDX)

    if len(tokens) < MAX_SEQ_LEN:
        tokens += [PAD_IDX] * (MAX_SEQ_LEN - len(tokens))
    else:
        tokens = tokens[:MAX_SEQ_LEN]
        tokens[-1] = EOS_IDX

    return torch.tensor(tokens, dtype=torch.long)


def decode_tokens(tokens):
    chars = []
    for t in tokens:
        t = int(t)
        if t == EOS_IDX:
            break
        if t in [PAD_IDX, SOS_IDX]:
            continue
        chars.append(idx2char.get(t, ""))
    return "".join(chars)

# ============================================================
# CUSTOM DATASET
# ============================================================

class HistoricalLineDataset(Dataset):
    def __init__(self, samples, augment=False):
        self.samples = samples
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, text = self.samples[idx]
        image = preprocess_image(img_path, augment=self.augment)
        tokens = encode_text(text)
        return image, tokens, text

# ============================================================
# DATALOADERS
# ============================================================

train_loader = DataLoader(
    HistoricalLineDataset(train_samples, augment=True), # Augmentations Active
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

val_loader = DataLoader(
    HistoricalLineDataset(val_samples, augment=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    HistoricalLineDataset(test_samples, augment=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

# ============================================================
# POSITIONAL BRIDGE
# ============================================================

class PositionalBridge(nn.Module):
    def __init__(self, in_channels=1024, d_model=768, vis_seq_len=256):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d((1, vis_seq_len))
        self.proj = nn.Linear(in_channels, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, vis_seq_len, d_model) * 0.02)

    def forward(self, x):
        B, H, W, C = x.shape
        x = x.permute(0, 3, 1, 2)    
        x = self.pool(x)             
        x = x.squeeze(2)             
        x = x.permute(0, 2, 1)       
        x = self.proj(x)             
        x = x + self.pos_embed
        return x

# ============================================================
# HISTORICAL DECODER BLOCK
# ============================================================

class HistoricalDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=768, n_heads=12, n_layers=6):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_embed = nn.Embedding(MAX_SEQ_LEN, d_model)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=3072,
            dropout=0.1,
            batch_first=True,
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(d_model, vocab_size)

        print("Initializing Decoder with Multilingual XLM-RoBERTa Embeddings...")
        try:
            xlm = XLMRobertaModel.from_pretrained("xlm-roberta-base")
            emb = xlm.embeddings.word_embeddings.weight
            n = min(emb.shape[0], vocab_size)
            self.token_embed.weight.data[:n] = emb[:n]
            del xlm
            print("Loaded pretrained embedding alignment successfully.")
        except Exception as e:
            print("Pretrained base load skipped/failed:", e)

    def forward(self, memory, tgt_tokens):
        B, T = tgt_tokens.shape
        pos = torch.arange(T, device=tgt_tokens.device).unsqueeze(0).expand(B, -1)
        x = self.token_embed(tgt_tokens) + self.pos_embed(pos)

        tgt_mask = torch.triu(torch.ones(T, T, device=tgt_tokens.device), diagonal=1).bool()
        tgt_key_padding_mask = (tgt_tokens == PAD_IDX)

        out = self.decoder(
            tgt=x,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
        )
        return self.output_proj(out)

    @torch.no_grad()
    def greedy_decode(self, memory, max_len=MAX_SEQ_LEN):
        B = memory.shape[0]
        generated = torch.full((B, 1), SOS_IDX, device=memory.device, dtype=torch.long)
        finished = torch.zeros(B, dtype=torch.bool, device=memory.device)

        for _ in range(max_len):
            logits = self.forward(memory, generated)
            next_token = logits[:, -1].argmax(dim=-1)
            next_token = next_token.masked_fill(finished, PAD_IDX)

            generated = torch.cat([generated, next_token.unsqueeze(1)], dim=1)
            finished |= (next_token == EOS_IDX)

            if finished.all():
                break
        return generated[:, 1:]

# ============================================================
# CORE MODEL ENGINE
# ============================================================

class HVLT(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(32, 192),
            strict_img_size=False,
        )
        self.bridge = PositionalBridge(in_channels=1024, d_model=768, vis_seq_len=256)
        self.decoder = HistoricalDecoder(vocab_size=VOCAB_SIZE)

    def forward(self, images, tgt_tokens):
        feats = self.encoder(images)[-1]  
        memory = self.bridge(feats)
        return self.decoder(memory, tgt_tokens)

    @torch.no_grad()
    def predict(self, images):
        feats = self.encoder(images)[-1]
        memory = self.bridge(feats)
        return self.decoder.greedy_decode(memory)

# ============================================================
# OPTIMIZATION CRITERIA
# ============================================================

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

def char_accuracy(preds, labels):
    correct, total = 0, 0
    for p, l in zip(preds, labels):
        n = min(len(p), len(l))
        for i in range(n):
            if p[i] == l[i]:
                correct += 1
        total += max(len(p), len(l))
    return 100.0 * correct / max(total, 1)

# ============================================================
# ALLOCATION
# ============================================================

model = HVLT().to(DEVICE)
optimizer = Adam(model.parameters(), lr=LR)
scaler = GradScaler("cuda", enabled=USE_AMP)

print("TOTAL INSTANTIATED PARAMS:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

# ============================================================
# CORE RUN LOOP
# ============================================================

best_val_loss = float('inf')  # Track validation loss instead of wild early WERs
patience_counter = 0  

for epoch in range(1, MAX_EPOCHS + 1):
    # --- TRAINING STAGE ---
    model.train()
    train_loss = 0.0
    train_preds, train_labels = [], []
    
    pbar = tqdm(train_loader)
    for images, targets, labels in pbar:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        decoder_input = targets[:, :-1]
        decoder_target = targets[:, 1:]

        optimizer.zero_grad()

        with autocast("cuda", enabled=USE_AMP):
            logits = model(images, decoder_input)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), decoder_target.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

        with torch.no_grad():
            pred_ids = logits.argmax(dim=-1)
            preds = [decode_tokens(x) for x in pred_ids]
            
        train_preds.extend(preds)
        train_labels.extend(labels)
        pbar.set_description(f"Epoch {epoch} | Train Loss: {loss.item():.4f}")

    train_cer = char_accuracy(train_preds, train_labels)
    train_wer_score = wer(train_labels, train_preds) * 100
    avg_train_loss = train_loss / len(train_loader)

    # --- VALIDATION STAGE ---
    model.eval()
    val_loss = 0.0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for images, targets, labels in tqdm(val_loader, desc="Validating"):
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            decoder_input = targets[:, :-1]
            decoder_target = targets[:, 1:]

            logits = model(images, decoder_input)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), decoder_target.reshape(-1))
            val_loss += loss.item()

            pred_ids = model.predict(images)
            preds = [decode_tokens(x) for x in pred_ids]
            
            val_preds.extend(preds)
            val_labels.extend(labels)

    val_cer = char_accuracy(val_preds, val_labels)
    val_wer_score = wer(val_labels, val_preds) * 100
    avg_val_loss = val_loss / len(val_loader)

    # --- METRIC LOGGING ---
    print("\n" + "="*60)
    print(f"EPOCH {epoch}")
    print(f"TRAIN LOSS: {avg_train_loss:.4f} | VAL LOSS: {avg_val_loss:.4f}")
    print(f"TRAIN CAR:  {train_cer:.2f}%  | VAL CAR:  {val_cer:.2f}%")
    print(f"TRAIN WER:  {train_wer_score:.2f}%  | VAL WER:  {val_wer_score:.2f}%")
    print("="*60)

    # --- STABLE SYSTEM EARLY STOPPING TRIGGER (MONITORS VAL LOSS) ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0  # Reset
        torch.save({
            "model": model.state_dict(),
            "epoch": epoch,
            "val_loss": avg_val_loss,
            "val_wer": val_wer_score,
        }, "best_icdar2017_train_a_hvlt.pt")
        print(f">>> PERFORMANCE INCREASE DETECTED: VAL LOSS DROPPED TO {avg_val_loss:.4f}. SAVED MODEL.")
    else:
        patience_counter += 1
        print(f">>> No improvement in Validation Loss. Early Stopping Counter: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"\n[EARLY STOPPING TRIGGERED] Validation metric stalled for {PATIENCE} epochs. Halting training process.")
        break

# ============================================================
# EVALUATION METRIC TESTING RUN
# ============================================================

print("\nLoading Best Model Weights for Final Inference Testing...")
if os.path.exists("best_icdar2017_hvlt.pt"):
    checkpoint = torch.load("best_icdar2017_hvlt.pt")
    model.load_state_dict(checkpoint["model"])
    print(f"Successfully reloaded checkpoints from epoch {checkpoint['epoch']}")

print("\nExecuting Final Generalization Performance Run on Independent Test Split...")
model.eval()
test_preds, test_labels = [], []

with torch.no_grad():
    for images, _, labels in tqdm(test_loader, desc="Testing Evaluation"):
        images = images.to(DEVICE)
        pred_ids = model.predict(images)
        preds = [decode_tokens(x) for x in pred_ids]
        test_preds.extend(preds)
        test_labels.extend(labels)

test_cer = char_accuracy(test_preds, test_labels)
test_wer_score = wer(test_labels, test_preds) * 100

print("\n" + "#"*40 + "\nFINAL INDEPENDENT TARGET EVALUATION METRICS")
print(f"TEST CHAR ACCURACY (CAR): {test_cer:.2f}%")
print(f"TEST WORD ERROR RATE (WER): {test_wer_score:.2f}%")
print("#"*40)

VOCAB SIZE: 102
TRAIN SEGMENTS: 793
VAL SEGMENTS:   137
TEST SEGMENTS:  152
Initializing Decoder with Multilingual XLM-RoBERTa Embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded pretrained embedding alignment successfully.
TOTAL INSTANTIATED PARAMS: 144.589694 M


Epoch 1 | Train Loss: 3.2280: 100%|██████████████████████████████████████████████████████████████| 25/25 [00:04<00:00,  5.95it/s]
Validating: 100%|██████████████████████████████████████████████████████████████████████████████████| 5/5 [00:04<00:00,  1.18it/s]


...


EPOCH 36
TRAIN LOSS: 0.9237 | VAL LOSS: 3.4762
TRAIN CAR:  50.47%  | VAL CAR:  5.92%
TRAIN WER:  89.99%  | VAL WER:  111.04%
>>> No improvement in Validation Loss. Early Stopping Counter: 19/20


Epoch 37 | Train Loss: 0.8917: 100%|█████████████████████████████████████████████████████████████| 25/25 [00:03<00:00,  6.67it/s]
Validating: 100%|██████████████████████████████████████████████████████████████████████████████████| 5/5 [00:04<00:00,  1.15it/s]



EPOCH 37
TRAIN LOSS: 0.8393 | VAL LOSS: 3.5761
TRAIN CAR:  51.53%  | VAL CAR:  6.37%
TRAIN WER:  87.13%  | VAL WER:  112.17%
>>> No improvement in Validation Loss. Early Stopping Counter: 20/20

[EARLY STOPPING TRIGGERED] Validation metric stalled for 20 epochs. Halting training process.

Loading Best Model Weights for Final Inference Testing...
Successfully reloaded checkpoints from epoch 1

Executing Final Generalization Performance Run on Independent Test Split...


Testing Evaluation: 100%|██████████████████████████████████████████████████████████████████████████| 5/5 [00:04<00:00,  1.06it/s]


########################################
FINAL INDEPENDENT TARGET EVALUATION METRICS
TEST CHAR ACCURACY (CAR): 10.80%
TEST WORD ERROR RATE (WER): 100.75%
########################################
